In [1]:
# 학번과 이름을 출력하세요
# 예시
# 학번 = '30261240'
# 이름 = '이현정'

학번 = '20242028'
이름 = '유건'

print(학번, 이름)

20242028 유건


In [2]:
class MyStack:
    def pop(self):
        return "Removed"

def remove_last(obj):
    return obj.pop()

print(remove_last([1,2,3]))
print(remove_last(MyStack()))

3
Removed


In [3]:
class MySequence:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

seq = MySequence([10,20,30])

print(len(seq))
print(seq[1])

for item in seq:
    print(item)

3
20
10
20
30


In [4]:
class Foo:
    def __getitem__(self, pos):
        return range(0, 30, 10)[pos]


f = Foo()

print(f[1])

for i in f:
    print(i)

print(20 in f)
print(15 in f)

10
0
10
20
True
False


In [5]:
PyObject *
PyObject_GetIter(PyObject *o)
{
    PyTypeObject *t = Py_TYPE(o);
    getiterfunc f;

    f = t->tp_iter;

    if (f == NULL) {
        if (PySequence_Check(o))
            return PySeqIter_New(o);

        return type_error(
            "'%.20s' object is not iterable",
            o
        );
    }
    else {
        PyObject *res = (*f)(o);

        if (res != NULL && !PyIter_Check(res)) {
            PyErr_Format(
                PyExc_TypeError,
                "iter() returned non-iterator"
            );

            Py_SETREF(res, NULL);
        }

        return res;
    }
}

SyntaxError: invalid syntax (180281787.py, line 1)

In [6]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])


class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [
            Card(rank, suit)
            for suit in self.suits
            for rank in self.ranks
        ]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

In [7]:
from random import shuffle

deck = FrenchDeck()

shuffle(deck)

TypeError: 'FrenchDeck' object does not support item assignment

In [8]:
from random import shuffle


def set_card(deck, position, card):
    deck._cards[position] = card


FrenchDeck.__setitem__ = set_card

shuffle(deck)

print(deck[:5])

[Card(rank='7', suit='spades'), Card(rank='Q', suit='hearts'), Card(rank='K', suit='hearts'), Card(rank='4', suit='diamonds'), Card(rank='2', suit='clubs')]


In [9]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])


class FrenchDeck2(collections.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [
            Card(rank, suit)
            for suit in self.suits
            for rank in self.ranks
        ]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, position, value):
        self._cards[position] = value

    def __delitem__(self, position):
        del self._cards[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

AttributeError: module 'collections' has no attribute 'MutableSequence'

In [10]:
from abc import ABC, abstractmethod


class Drawable(ABC):
    @abstractmethod
    def draw(self):
        pass


# 수정할 수 없는 외부 클래스라고 가정
class ExternalPainter:
    def draw(self):
        print("Drawing")


Drawable.register(ExternalPainter)

obj = ExternalPainter()

print(isinstance(obj, Drawable))

True


In [11]:
from collections.abc import Sized


class MyCollection:
    def __len__(self):
        return 10


Sized.register(MyCollection)

__main__.MyCollection

In [ ]:
import abc


class Tombola(abc.ABC):

    @abc.abstractmethod
    def load(self, iterable):
        pass

    @abc.abstractmethod
    def pick(self):

        pass

    def loaded(self):
        return bool(self.inspect())

    def inspect(self):
        items = []

        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break

        self.load(items)

        return tuple(sorted(items))

In [16]:
import random

from tombola import Tombola


class BingoCage(Tombola):

    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        self.pick()

In [17]:
import random

from tombola import Tombola


class LotteryBlower(Tombola):

    def __init__(self, iterable):
        self._balls = list(iterable)

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty LotteryBlower')

        return self._balls.pop(position)

    def loaded(self):
        return bool(self._balls)

    def inspect(self):
        return tuple(sorted(self._balls))

In [18]:
from random import randrange

from tombola import Tombola


@Tombola.register
class TomboList(list):

    def pick(self):
        if self:
            position = randrange(len(self))
            return self.pop(position)
        else:
            raise LookupError('pop from empty TomboList')

    load = list.extend

    def loaded(self):
        return bool(self)

    def inspect(self):
        return tuple(sorted(self))

In [24]:
import doctest

from tombola import Tombola

# modules to test
import bingo
import lotto
import tombolist
import drum


TEST_FILE = 'tombola_tests.rst'
TEST_MSG = '{0:16} {1.attempted:2} tests, {1.failed:2} failed - {2}'


def main(argv):
    verbose = '-v' in argv

    real_subclasses = Tombola.__subclasses__()
    virtual_subclasses = list(Tombola._abc_registry)

    for cls in real_subclasses + virtual_subclasses:
        test(cls, verbose)


def test(cls, verbose=False):
    res = doctest.testfile(
        TEST_FILE,
        globs={'ConcreteTombola': cls},
        verbose=verbose,
        optionflags=doctest.REPORT_ONLY_FIRST_FAILURE
    )

    tag = 'FAIL' if res.failed else 'OK'

    print(TEST_MSG.format(cls.__name__, res, tag))


if __name__ == '__main__':
    import sys

    main(sys.argv)

ModuleNotFoundError: No module named 'drum'

In [25]:
from tombola import Tombola
import bingo
import lotto
import tombolist

print("모듈 불러오기 성공!")

모듈 불러오기 성공!
